## Implement PKCS#7 padding

In [ ]:
pt = "YELLOW SUBMARINE"
offset = abs(len(pt) - 20)
ct = pt.encode() + (offset.to_bytes() * offset)
print(ct)

## Implement CBC mode

In [ ]:
import base64
from Crypto.Cipher import AES

In [ ]:
def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

In [ ]:
with open("/mnt/d/my-kisah/crypto/1.cryptopals/files/10.txt") as f:
    b64_data = "".join(
        line.strip()
        for line in f
    )

ct = base64.b64decode(b64_data)
key = b"YELLOW SUBMARINE"
iv  = (len(key)*0).to_bytes() * len(key)
blocks = [ct[i:i+16] for i in range(0,len(ct),16)]

pt = b""
cipher = AES.new(key, AES.MODE_ECB)
for i in range(len(blocks)):
    ci = cipher.decrypt(blocks[i])
    if i == 0:
        pt += fixed_xor(ci, iv)
        continue
    pt += fixed_xor(ci, blocks[i-1])

print(pt.decode())    

## An ECB/CBC detection oracle


In [ ]:
import os
import random
from Crypto.Cipher import AES

In [ ]:
def salting(pt: bytes):
    count = random.randrange(5,11)
    prefix = os.urandom(count)
    suffix = os.urandom(count)
    return prefix + pt + suffix

def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def fixed_xor(b1: bytes, b2: bytes):
    if len(b1) - len(b2) != 0:
        raise Exception("buffer must have same length")
    return bytes(bytearray([x ^ y for x, y in zip(b1, b2)]))

def aes_ecb(pt: bytes, key: bytes):
    plain = b""
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def aes_cbc(pt: bytes, key: bytes):
    iv = os.urandom(16)
    plain = salting(pt)
    if len(plain) % 16 != 0:
        plain = pkcs7(plain)
    
    blocks = [plain[i:i+16] for i in range(0,len(plain),16)]
    cipher = AES.new(key, AES.MODE_ECB)
    for i in range(len(blocks)):
        if i == 0:
            ci = fixed_xor(blocks[i], iv)
            blocks[i] = cipher.encrypt(ci)
            continue
        ci = fixed_xor(blocks[i-1], blocks[i])
        blocks[i] = cipher.encrypt(ci)
    
    ct = b"".join(blocks)
    return ct

In [ ]:
def aes_oracle(pt: str):
    enc_pt = pt.encode()
    
    key = os.urandom(16)
    aes_mode = random.randrange(1,3)
    ct = b""
    match aes_mode:
        case 1:
            # print("ECB")
            ct = aes_ecb(enc_pt, key)
        case 2:
            # print("CBC")
            ct = aes_cbc(enc_pt, key)
    return ct

In [ ]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

In [ ]:
pt = "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"
ct = aes_oracle(pt)
print(ct)

blocks = split_block(ct)
repeated = check_repeated(blocks)
if repeated > 0:
    print(f"Its ECB")
else:
    print("Its CBC")

## Byte-at-a-time ECB decryption (Simple)
 

In [ ]:
import os
import random
import base64
from Crypto.Cipher import AES

KEY = os.urandom(16)

In [ ]:
def salting(pt: bytes):
    salt = "Um9sbGluJyBpbiBteSA1LjAKV2l0aCBteSByYWctdG9wIGRvd24gc28gbXkgaGFpciBjYW4gYmxvdwpUaGUgZ2lybGllcyBvbiBzdGFuZGJ5IHdhdmluZyBqdXN0IHRvIHNheSBoaQpEaWQgeW91IHN0b3A/IE5vLCBJIGp1c3QgZHJvdmUgYnkK"
    salt_enc = base64.b64decode(salt)
    return pt + salt_enc

def pkcs7(pt:bytes):
    offset = 16 - (len(pt)%16)
    ct = pt + (offset.to_bytes() * offset)
    return ct

def aes_ecb(pt: bytes, key: bytes):
    plain = b""
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def aes_oracle(pt: bytes):
    enc_pt = pt
    salting_pt = salting(enc_pt)
    key = KEY
    ct = ct = aes_ecb(salting_pt, key)
    return ct

In [ ]:
def split_block(ciphertext: bytes, size = 16):
    blocks = []
    for i in range(0, len(ciphertext), size):
        blocks.append(ciphertext[i:i+size])
    return blocks

def check_repeated(block: list) -> int:
    block_length = len(block)
    uniq_block = len(set(block))
    diff = abs(block_length - uniq_block)
    if diff != 0:
        return diff
    return 0

def find_block_size():
    base_len = 0
    for i in range(0,64):
        pt = b'A'*i
        ct = aes_oracle(pt)
        if i == 0:
            base_len += len(ct)
            continue
        if len(ct) > base_len:
            return len(ct) - base_len
    raise Exception("Block size not found")

In [ ]:
block_size = find_block_size()
print(f"Block size: {block_size}")

In [ ]:
pt = b'A'*(block_size*3)
ct = aes_oracle(pt)
blocks = split_block(ct, block_size)
repeated = check_repeated(blocks)

if repeated > 0:
    print("its ECB")

In [ ]:
sentence_cipher = {}
for w in range(256):
    pt = b'A'*15 + bytes([w])
    ct = aes_oracle(pt)
    sentence_cipher[ct[:16]] = bytes([w])

# for k in sentence_cipher:
#     print(k, sentence_cipher[k])

In [ ]:
short_ct = aes_oracle(b'A'*15)[:16]
first_byte = sentence_cipher[short_ct]
print(f"Byte secret pertama: {first_byte}")

In [ ]:
known_pt = b""
while True:
    pad_len = (block_size - 1) - (len(known_pt) % block_size)
    prefix = b"A" * pad_len
    target_block_idx = len(known_pt) // block_size

    sentence_cipher = {}
    for w in range(256):
        cand = prefix + known_pt + bytes([w])
        ct = aes_oracle(cand)
        sentence_cipher[ct[:16]] = bytes([w])
        dict_key = split_block(ct, block_size)[target_block_idx]
        sentence_cipher[dict_key] = bytes([w])

    ct_short = aes_oracle(prefix)
    target_ct_block = split_block(ct_short, block_size)[target_block_idx]

    if target_ct_block in sentence_cipher:
        known_pt += sentence_cipher[target_ct_block]
    else:
        break 
print(known_pt.decode())

## ECB cut-and-paste


In [ ]:
routine = "foo=bar&baz=qux&zap=zazzle"
raw = (routine.replace("&","=")).split("=")
dictionary = {}
for i in range(len(raw)):
    if i % 2 == 0:
        # print(raw[i])
        dictionary[raw[i]] = raw[i+1]

print(dictionary)

In [ ]:
KEY = os.urandom(16)

In [ ]:
def profile_for(profile: str):
    email = profile.replace("&","").replace("=","")
    uid = 10
    role = 'user'

    return f"email={email}&uid={uid}&role={role}"

def pkcs7(pt:bytes, block_size: int = 16):
    offset = block_size - (len(pt)%block_size)
    return pt + (offset.to_bytes() * offset)

def encrypt_profile(pt: bytes):
    plain = b""
    key = KEY
    if len(pt) % 16 != 0:
        plain = pkcs7(pt)
    else:
        plain = pt
    cipher = AES.new(key, AES.MODE_ECB)
    ct = cipher.encrypt(plain)
    return ct

def decrypt_profile(profile_enc: bytes):
    cipher = AES.new(KEY, AES.MODE_ECB)
    dec = cipher.decrypt(profile_enc)
    pad_len = dec[-1]
    return dec[:-pad_len]

def parse_cookies(dec: bytes):
    routine = dec.decode()
    raw = (routine.replace("&","=")).split("=")
    dictionary = {}
    for i in range(len(raw)):
        if i % 2 == 0:
            # print(raw[i])
            dictionary[raw[i]] = raw[i+1]
    return dictionary

profile = profile_for("foo@bar.com")
encrypt = encrypt_profile(profile.encode())
decrypt = decrypt_profile(encrypt)
cookies = parse_cookies(decrypt)
print(encrypt)
print(decrypt)
print(cookies)

email=      -> 6
email       -> x
&uid=       -> 5
uid         -> 2
&role=      -> 6
total = 19

x + 19 ≡ 0 (mod 16)
x = 13
normal ct
block1 = email=AAAAA@bar.com
block2 = &uid=10&role=user+padd

spoof ct
block1 = email=admin

In [ ]:
email = "AAAAA@bar.com"
profile = profile_for(email)
encrypt = encrypt_profile(profile.encode())
print(profile[:32])
print(encrypt)
print()

payload = 'B'*10+"admin" + '\x0b'*11 
profile_admin = profile_for(payload)
encrypt_admin = encrypt_profile(profile_admin.encode())
crafted = encrypt[:32] + encrypt_admin[16:32]
decrypt_admin = decrypt_profile(crafted)
cookies_admin = parse_cookies(decrypt_admin)
print(cookies_admin)

## Byte-at-a-time ECB decryption (Harder)